# Hybrid Search and Reranking in RAG

## What This Notebook Covers

This notebook builds a production-grade Retrieval Augmented Generation pipeline that combines three retrieval strategies and one reranking step.

The pipeline is organised into five clearly separated categories.

Category 1 handles all environment setup including loading secrets, connecting to the Weaviate cloud vector database, and defining the schema that tells Weaviate how to store and vectorise documents.

Category 2 loads the local TinyLlama language model, its tokenizer, and wraps everything into a LangChain-compatible LLM object ready for inference.

Category 3 handles document ingestion by loading the source PDF, splitting it into chunks, sanitising metadata so Weaviate accepts it, and uploading all chunks to the remote vector store.

Category 4 runs hybrid retrieval and demonstrates two ways to build a RAG chain using Weaviate as the backend, and also shows the LCEL (LangChain Expression Language) style chain.

Category 5 adds a Cohere reranker on top of the hybrid retriever so retrieved candidates are reordered by a dedicated cross-encoder model before being passed to the LLM, producing higher-quality answers.

## Why Each Layer Exists

Weaviate is a cloud-hosted vector database that stores document embeddings and supports hybrid search natively. Using a managed cloud database means you do not need to run ChromaDB locally and the index persists between sessions.

Hybrid search in Weaviate is controlled by the alpha parameter. An alpha of 0 is pure BM25 keyword search. An alpha of 1 is pure dense vector search. An alpha of 0.5 blends both signals equally which is what this notebook uses.

Reranking is a second-pass scoring step. The initial retriever fetches a broad set of candidates quickly using approximate methods. A reranker then re-scores every candidate using a more expensive cross-encoder model that reads the query and each document together, which produces much more accurate relevance scores than the first-pass retriever alone.

## Category 1  Environment Setup and Weaviate Connection

### Loading Secrets Safely

API keys and database URLs must never be hard-coded into a notebook. If you push hard-coded secrets to GitHub, anyone who finds the repository can use your paid cloud services at your expense.

The python-dotenv library solves this by reading a plain text file named .env from your local disk. The .env file is listed in .gitignore so it never gets committed. Inside the notebook you call load_dotenv which reads that file and injects the key-value pairs as environment variables. Then os.getenv retrieves each variable by name.

The override=True argument tells load_dotenv to overwrite any environment variable that was already set in the shell session. This ensures the notebook always uses the values from the .env file rather than stale values from a previous run.

In [1]:
from dotenv import load_dotenv
import os

# Load secrets from the .env file on disk into the process environment
# override=True ensures fresh values from .env replace any previously set shell variables
load_dotenv(r"F:\Advance RAG\.env", override=True)

# Read each secret by name from the environment
# WEAVIATE_CLUSTER: the full URL of your Weaviate cloud cluster, e.g. https://xxxx.weaviate.network
# WEAVIATE_API_KEY: the API key that authenticates your requests to that cluster
# HF_TOKEN: your HuggingFace API token, needed so Weaviate can call HuggingFace models for vectorisation
WEAVIATE_URL = os.getenv("WEAVIATE_CLUSTER")
WEAVIATE_API_KEY = os.getenv("WEAVIATE_API_KEY")
HF_TOKEN = os.getenv("HF_TOKEN")

In [2]:
# Verify the URL was loaded correctly by printing its repr
# repr shows invisible characters and helps spot missing newlines or extra spaces
print(repr(WEAVIATE_URL))

'https://weotj5k5q1mzslz0pq0m9w.c0.eu-central-1.aws.weaviate.cloud'


### Connecting to the Weaviate Cloud Cluster

Weaviate is an open-source vector database that can run locally or on their managed cloud service (WCS). This notebook uses the cloud version so the index persists between sessions and no local server is required.

The client object is the single entry point for all Weaviate operations: creating schemas, uploading documents, and running queries. Authentication uses AuthApiKey which sends your API key as a bearer token on every request. The additional_headers dictionary passes your HuggingFace token so Weaviate can call the HuggingFace Inference API to vectorise text on your behalf.

In [3]:
import weaviate

# Create a Weaviate client connected to your cloud cluster
# url: the full HTTPS address of your Weaviate cloud instance
# auth_client_secret: authenticates you with your API key so only you can read and write your data
# additional_headers: forwarded to external API calls Weaviate makes on your behalf
#   X-HuggingFace-Api-Key lets Weaviate call HuggingFace text2vec models to embed your documents
client = weaviate.Client(
    url=WEAVIATE_URL,
    auth_client_secret=weaviate.AuthApiKey(WEAVIATE_API_KEY),
    additional_headers={
        "X-HuggingFace-Api-Key": HF_TOKEN
    }
)

In [4]:
# Verify the client can reach the cluster
# Returns True if the cluster is up and reachable, False otherwise
# If this returns False, check your WEAVIATE_URL value and network connectivity
client.is_ready()

True

### Resetting the Schema Before Creating a Fresh One

Before creating the RAG class we delete it if it already exists. This gives a clean slate and avoids errors from duplicate class names or schema conflicts between runs.

In production you would only do this during development. In a real deployed system you would add documents without deleting the existing index.

In [5]:
# Delete the RAG class and all documents stored in it if it already exists
# This is a destructive operation that cannot be undone
# It ensures we start with a clean schema on every fresh run of this notebook
client.schema.delete_class("RAG")

### Defining the Weaviate Schema

A schema in Weaviate describes a class which is analogous to a table in a relational database. The schema specifies three things: what properties each document has, which vectoriser module should embed the text, and which embedding model that module should use.

The vectoriser is set to text2vec-huggingface which tells Weaviate to call the HuggingFace Inference API to embed every document property that has skip set to False. The model is sentence-transformers/all-MiniLM-L6-v2, a fast and widely used sentence embedding model that produces 384-dimensional vectors.

The content property is the only property we store and it will be vectorised. Setting vectorizePropertyName to False means the property name itself is not included in the embedding, only the property value.

In [6]:
schema = {
    "classes": [
        {
            "class": "RAG",
            "description": "Documents for RAG",

            # text2vec-huggingface tells Weaviate to use HuggingFace for embedding
            # Weaviate calls the HF Inference API using the X-HuggingFace-Api-Key header
            "vectorizer": "text2vec-huggingface",
            "moduleConfig": {
                "text2vec-huggingface": {
                    "model": "sentence-transformers/all-MiniLM-L6-v2",
                    "type": "text"
                }
            },

            "properties": [
                {
                    "dataType": ["text"],
                    "description": "The content of the paragraph",
                    "moduleConfig": {
                        "text2vec-huggingface": {
                            # skip=False means this property WILL be vectorised
                            # skip=True would store the text but not embed it
                            "skip": False,
                            # vectorizePropertyName=False means the word "content"
                            # is not included in the embedding, only the actual text value
                            "vectorizePropertyName": False,
                        }
                    },
                    "name": "content",
                },
            ],
        },
    ]
}

In [7]:
# Send the schema definition to Weaviate
# After this call, the RAG class exists in the cluster and is ready to accept documents
client.schema.create(schema)

In [8]:
# Inspect the schema that was just created to confirm it matches the definition above
client.schema.get()

{'classes': [{'class': 'RAG',
   'description': 'Documents for RAG',
   'invertedIndexConfig': {'bm25': {'b': 0.75, 'k1': 1.2},
    'cleanupIntervalSeconds': 60,
    'stopwords': {'additions': None, 'preset': 'en', 'removals': None},
    'usingBlockMaxWAND': True},
   'moduleConfig': {'text2vec-huggingface': {'model': 'sentence-transformers/all-MiniLM-L6-v2',
     'type': 'text',
     'useCache': True,
     'useGPU': False,
     'vectorizeClassName': True,
     'waitForModel': False}},
   'multiTenancyConfig': {'autoTenantActivation': False,
    'autoTenantCreation': False,
    'enabled': False},
   'properties': [{'dataType': ['text'],
     'description': 'The content of the paragraph',
     'indexFilterable': True,
     'indexRangeFilters': False,
     'indexSearchable': True,
     'moduleConfig': {'text2vec-huggingface': {'skip': False,
       'vectorizePropertyName': False}},
     'name': 'content',
     'tokenization': 'word'}],
   'shardingConfig': {'actualCount': 1,
    'actualV

### Initialising the Weaviate Hybrid Search Retriever

WeaviateHybridSearchRetriever wraps the Weaviate client inside the standard LangChain retriever interface. When you call invoke on this retriever it sends a hybrid search request to your Weaviate cluster that combines BM25 keyword scoring and dense vector cosine similarity scoring.

The alpha parameter is the most important setting here. It controls the balance between keyword search and semantic search using this formula:

hybrid_score equals (1 minus alpha) multiplied by sparse_score plus alpha multiplied by dense_score

With alpha set to 0.5 both signals contribute equally. Increase alpha toward 1.0 to rely more on meaning-based similarity. Decrease it toward 0.0 to rely more on exact keyword matching.

The k parameter controls how many documents are returned per query. Setting it to 2 here keeps the retrieved context short enough for TinyLlama which has a small context window.

In [9]:
from langchain_community.retrievers import WeaviateHybridSearchRetriever

# Create the hybrid retriever connected to our Weaviate cluster
# client: the authenticated Weaviate client object
# index_name: the Weaviate class to search, must match the class we created in the schema
# text_key: the property name that stores the document text inside each Weaviate object
# alpha=0.5: equal weight to BM25 keyword scoring and dense vector semantic scoring
# k=2: return the top 2 most relevant chunks per query
# attributes=[]: no extra metadata properties to return, only the text_key content
# create_schema_if_missing=False: do not auto-create a schema, we already created it explicitly above
retriever = WeaviateHybridSearchRetriever(
    client=client,
    index_name="RAG",
    text_key="content",
    alpha=0.5,
    k=2,
    attributes=[],
    create_schema_if_missing=False,
)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_11924\110612868.py:11: LangChainDeprecationWarning: The class `WeaviateHybridSearchRetriever` was deprecated in LangChain 0.3.18 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-weaviate package and should be used instead. To use it run `pip install -U :class:`~langchain-weaviate` and import as `from :class:`~langchain_weaviate import WeaviateVectorStore``.
  retriever = WeaviateHybridSearchRetriever(


## Category 2  Language Model Setup

### Why We Load the LLM Before Ingesting Documents

The document ingestion step can take several minutes because it embeds and uploads every chunk. Loading the model first ensures that if the LLM setup fails (for example due to insufficient memory or a missing package) you find out immediately rather than after waiting for a long ingestion run.

### About TinyLlama

TinyLlama-1.1B-Chat is a compact 1.1 billion parameter decoder-only transformer fine-tuned for instruction following and chat. It runs on a standard laptop CPU without any quantization tricks. The trade-off is that its context window and reasoning capacity are limited compared to larger models, which is why max_new_tokens is kept small in this notebook.

In [10]:
from transformers import (
    AutoModelForCausalLM,   # Loads any decoder-only causal language model from HuggingFace Hub
    AutoTokenizer,           # Loads the paired tokenizer for any HuggingFace model
    pipeline                 # High-level API combining model and tokenizer into one inference object
)

from langchain_community.llms import HuggingFacePipeline  # Wraps an HF pipeline as a LangChain LLM

In [11]:
def load_model(model_name: str):
    # AutoModelForCausalLM.from_pretrained downloads the model weights from HuggingFace Hub
    # on the first run and uses the local cache on all subsequent runs
    # For TinyLlama this is approximately 2 GB of weights
    model = AutoModelForCausalLM.from_pretrained(
        model_name
    )
    return model

In [12]:
# TinyLlama-1.1B-Chat-v1.0 is instruction-tuned and small enough to run on CPU
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0" 

In [13]:
# Load the model weights from HuggingFace Hub into memory
# This downloads approximately 2 GB on the first run
model = load_model(model_name)

In [14]:
# initializing tokenizer
def initialize_tokenizer(model_name: str):
    # AutoTokenizer.from_pretrained loads the tokenizer saved alongside the model
    # return_token_type_ids=False: TinyLlama does not use token type IDs (a BERT-specific feature)
    # Disabling this avoids a harmless but confusing deprecation warning
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        return_token_type_ids=False
    )

    # bos_token_id=1 sets the beginning-of-sequence token explicitly
    # This tells the model where the start of every input begins
    tokenizer.bos_token_id = 1

    return tokenizer

In [15]:
# Initialise the tokenizer using the same model name as the model
# The tokenizer must match the model exactly so token IDs correspond correctly
tokenizer = initialize_tokenizer(model_name)

### Building the Text Generation Pipeline

The HuggingFace pipeline function wraps the model and tokenizer into a single object that handles the full inference loop: tokenising the input, running the forward pass, sampling the next token from the output distribution, appending it, and repeating until the end-of-sequence token is produced or max_new_tokens is reached.

do_sample=False means greedy decoding: at each step the model picks the single highest-probability next token deterministically. This produces the same output every time for the same input which is useful for reproducibility during development.

max_new_tokens=20 is very small and is appropriate for this notebook only because TinyLlama has a small context window and the retrieved chunks already occupy most of it. In a production system with a larger model you would set this to 256 or more.

In [37]:
from transformers import pipeline

text_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="cpu",          # Run entirely on CPU since TinyLlama is small enough
    do_sample=False,           # Greedy decoding: always pick the most probable next token
    max_new_tokens=100,         # Generate at most 20 new tokens beyond the input prompt
    use_cache=False,           # Disable KV cache to save memory on CPU
    num_return_sequences=1,    # Generate one response per call
    eos_token_id=tokenizer.eos_token_id,   # Stop when end-of-sequence token is produced
    pad_token_id=tokenizer.eos_token_id,   # Use EOS token as padding token for batched inputs
)

Device set to use cpu


In [17]:
# Wrap the HuggingFace pipeline as a LangChain LLM
# HuggingFacePipeline implements the standard LangChain LLM interface
# so this llm object can be plugged into any LangChain chain without modification
llm = HuggingFacePipeline(pipeline=text_pipeline)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_11924\1846957108.py:4: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=text_pipeline)


## Category 3  Document Ingestion

### The Full Ingestion Pipeline

Document ingestion transforms a raw PDF file into a set of searchable chunks stored inside the Weaviate cloud database. It involves four steps: loading the PDF, splitting it into chunks, sanitising metadata so Weaviate accepts it, and uploading all chunks to the remote vector store.

Each step is necessary. Without loading there is no text. Without splitting the chunks would exceed both the LLM context window and the Weaviate object size limit. Without sanitising metadata the upload fails because Weaviate property names cannot contain dots or hyphens. Without uploading nothing is searchable.

In [18]:
# Path to the PDF that will be our knowledge base
# Adjust this path to wherever you have stored the file
doc_path = r"../Data/Retrieval-Augmented-Generation-for-NLP.pdf" 

In [19]:
from langchain_community.document_loaders import PyPDFLoader

In [20]:
# Initialise the loader with the file path
# PyPDFLoader uses the pypdf library internally to parse the PDF binary format
loader = PyPDFLoader(doc_path)

In [21]:
# Load the PDF and receive a list of LangChain Document objects
# Each Document represents one page and has two attributes:
#   page_content: the raw text extracted from that page
#   metadata: a dict with keys like source (file path) and page (page number)
docs = loader.load()

In [22]:
# Inspect all loaded documents
docs

[Document(metadata={'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:a6404ea)', 'creationdate': '', 'author': 'Shangyu Wu; Ying Xiong; Yufei Cui; Haolun Wu; Can Chen; Ye Yuan; Lianming Huang; Xue Liu; Tei-Wei Kuo; Nan Guan; Chun Jason Xue', 'doi': 'https://doi.org/10.48550/arXiv.2407.13193', 'license': 'http://creativecommons.org/licenses/by/4.0/', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.28 (TeX Live 2025) kpathsea version 6.4.1', 'title': 'Retrieval-Augmented Generation for Natural Language Processing: A Survey', 'trapped': '/False', 'arxivid': 'https://arxiv.org/abs/2407.13193v4', 'source': '../Data/Retrieval-Augmented-Generation-for-NLP.pdf', 'total_pages': 67, 'page': 0, 'page_label': '1'}, page_content='Retrieval-Augmented Generation for Natural\nLanguage Processing: A Survey\nShangyu Wu1,2, Ying Xiong2†, Yufei Cui3, Haolun Wu3, Can\nChen3, Ye Yuan3, Lianming Huang1, Xue Liu2, Tei-Wei Kuo4,\nNan Guan1, Chun Jason Xue2\n1City University of Ho

In [23]:
# Inspect the first document to see what page_content and metadata look like
docs[0]

Document(metadata={'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:a6404ea)', 'creationdate': '', 'author': 'Shangyu Wu; Ying Xiong; Yufei Cui; Haolun Wu; Can Chen; Ye Yuan; Lianming Huang; Xue Liu; Tei-Wei Kuo; Nan Guan; Chun Jason Xue', 'doi': 'https://doi.org/10.48550/arXiv.2407.13193', 'license': 'http://creativecommons.org/licenses/by/4.0/', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.28 (TeX Live 2025) kpathsea version 6.4.1', 'title': 'Retrieval-Augmented Generation for Natural Language Processing: A Survey', 'trapped': '/False', 'arxivid': 'https://arxiv.org/abs/2407.13193v4', 'source': '../Data/Retrieval-Augmented-Generation-for-NLP.pdf', 'total_pages': 67, 'page': 0, 'page_label': '1'}, page_content='Retrieval-Augmented Generation for Natural\nLanguage Processing: A Survey\nShangyu Wu1,2, Ying Xiong2†, Yufei Cui3, Haolun Wu3, Can\nChen3, Ye Yuan3, Lianming Huang1, Xue Liu2, Tei-Wei Kuo4,\nNan Guan1, Chun Jason Xue2\n1City University of Hon

In [24]:
# Inspect just the metadata dictionary of the first page
# This shows what keys PyPDFLoader attaches by default
docs[0].metadata

{'producer': 'pikepdf 8.15.1',
 'creator': 'arXiv GenPDF (tex2pdf:a6404ea)',
 'creationdate': '',
 'author': 'Shangyu Wu; Ying Xiong; Yufei Cui; Haolun Wu; Can Chen; Ye Yuan; Lianming Huang; Xue Liu; Tei-Wei Kuo; Nan Guan; Chun Jason Xue',
 'doi': 'https://doi.org/10.48550/arXiv.2407.13193',
 'license': 'http://creativecommons.org/licenses/by/4.0/',
 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.28 (TeX Live 2025) kpathsea version 6.4.1',
 'title': 'Retrieval-Augmented Generation for Natural Language Processing: A Survey',
 'trapped': '/False',
 'arxivid': 'https://arxiv.org/abs/2407.13193v4',
 'source': '../Data/Retrieval-Augmented-Generation-for-NLP.pdf',
 'total_pages': 67,
 'page': 0,
 'page_label': '1'}

In [25]:
# Check how many pages were loaded from the PDF
len(docs)

67

### Splitting Documents Into Chunks

LLMs have fixed context window limits and embedding models have maximum sequence lengths. A full research paper page is typically 2000 to 4000 characters, far too long for either constraint. Splitting into smaller chunks solves both problems simultaneously.

chunk_size=500 produces chunks of at most 500 characters each. This is larger than in the hybrid search notebook because Weaviate handles larger objects efficiently and TinyLlama benefits from slightly more context per chunk.

chunk_overlap=50 ensures the last 50 characters of one chunk also appear at the start of the next chunk. This overlap prevents information from being invisibly cut off at chunk boundaries when a sentence or concept spans the split point.

In [26]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,    # Each chunk contains at most 500 characters
    chunk_overlap=50   # Consecutive chunks share 50 characters to preserve boundary context
)

split_docs = text_splitter.split_documents(docs)

In [27]:
# Verify how many chunks were produced from the full document
print(len(split_docs))

441


### Sanitising Metadata Keys for Weaviate

PyPDFLoader attaches metadata to every Document object. The default metadata keys include source and page. The source value is the file path which on Windows contains backslashes and on some systems contains dots or hyphens in folder names.

Weaviate property names must follow GraphQL naming rules: they can only contain letters, digits, and underscores. Any dot, hyphen, or space in a metadata key will cause Weaviate to reject the entire upload batch with an error.

This loop replaces every dot, hyphen, and space in every metadata key with an underscore across all chunks before upload.

In [28]:
for doc in split_docs:
    # Rebuild the metadata dictionary for every chunk
    # Replace dots, hyphens, and spaces in key names with underscores
    # so that Weaviate accepts every key as a valid GraphQL property name
    # Only the KEYS are changed, the VALUES are left exactly as they are
    doc.metadata = {
        k.replace(".", "_").replace("-", "_").replace(" ", "_"): v
        for k, v in doc.metadata.items()
    }

### Uploading Chunks to Weaviate

add_documents sends every chunk to the Weaviate cluster. For each chunk Weaviate internally calls the HuggingFace Inference API using the X-HuggingFace-Api-Key we provided when creating the client. The API embeds the chunk text using sentence-transformers/all-MiniLM-L6-v2 and stores both the raw text and the resulting 384-dimensional vector together.

This combined storage is what makes hybrid search possible. The keyword index (BM25) is built from the raw text. The vector index is built from the stored embeddings. Both indexes are queried simultaneously at retrieval time.

In [29]:
# Upload all chunks to the Weaviate cloud cluster
# For each chunk Weaviate calls HuggingFace to embed the text and stores both text and vector
# This may take a few minutes for a long document because of the HuggingFace API calls
retriever.add_documents(split_docs)

['9684155b-9d5f-480b-bc19-bf86d1a3d092',
 '70ded5ab-6d04-4e1d-8419-1b6dc7bc9c2c',
 '9f2c9962-9f0f-41b8-b740-12d4445ec240',
 '49763a43-c6de-4c6e-b165-1670a3b93890',
 'b9bb9bd6-17b6-40f8-a504-fe00e5dd2f38',
 '13397d9e-5d21-4267-a06a-a3fe5db8ddf4',
 '2f2db751-8471-4ad9-a9d4-9cd34c5ce9a5',
 '6cf120b0-9c1d-4aa1-8218-3e6d58982d5b',
 'b24852e9-f610-4cf0-9341-8f0559e126f2',
 'eb9754b3-1788-47e3-a10a-c5e22c4da288',
 'a67cfe8d-c9be-489a-81ff-6c5a797cbeb9',
 '49780ce3-5bd1-4299-b5d7-068ef6124093',
 '06df65ec-1097-4e5e-8e26-0e8ff982d19b',
 '72c5dc48-3256-4534-8b21-abf4e1bfcb3d',
 '9cabc6ba-fcd5-41b3-98ac-8263a2e4286b',
 '26946178-313a-46c8-b86e-9df258914565',
 '7b0f9d04-8d56-4abb-9e7f-f6ddccfea04f',
 'dbb8985f-6cc9-4579-af4b-c00b2b8c1b0d',
 '8f961d87-4bcd-4d4f-a8dd-6a39e86614cb',
 'e01b9b47-214f-4cdf-b584-548b39c99e91',
 '1e6ad449-5e11-4efa-928e-e33e4704e60f',
 '2b9d66bd-8211-4b59-9e43-42ab8f43a995',
 'db363a5c-c24e-40e5-8370-2a24172dddd0',
 'b059698e-b8cf-491a-97af-7247f49a3d84',
 'c71c3741-6e2e-

## Category 4  Hybrid Retrieval and RAG Chains

### Testing the Retriever Directly

Before building a full RAG chain it is good practice to test the retriever on its own. This confirms that documents were uploaded correctly and that Weaviate is returning relevant results before we add the complexity of an LLM on top.

In [30]:
# Test the hybrid retriever directly without any LLM involved
# invoke sends the query to Weaviate which runs hybrid search and returns k=2 Document objects
results = retriever.invoke("What is Retrieval-Augmented Generation?")

In [31]:
# Check how many documents were returned
# Should be 2 because we set k=2 in the retriever configuration
len(results)

2

In [32]:
# Read the text content of the most relevant retrieved chunk
# This lets you verify the retriever found genuinely relevant content
print(results[0].page_content)

cient retrieval and data persistence. A key design decision in the knowledge base is
what should be stored as values. For example, for question-answer tasks, when adding
retrievals to prompts, the naive but effective way is to store the question embedding
as the key and question-answer pairs as the value. This can help the generation pro-
cess as retrievals are used as demonstrations for models. Recent works have proposed


In [33]:
# Invoke with score=True to see the hybrid relevance scores alongside the documents
# Higher score means Weaviate considered the chunk more relevant to the query
retriever.invoke(
    "what is RAG token?",
    score=True
)

[Document(metadata={'_additional': {'explainScore': '\nHybrid (Result Set keyword,bm25) Document 6b894fa2-202f-4630-b746-d97a410a6720: original score 2.4784236, normalized score: 0.47461995 - \nHybrid (Result Set vector,hybridVector) Document 6b894fa2-202f-4630-b746-d97a410a6720: original score 0.5199056, normalized score: 0.5', 'score': '0.97462'}}, page_content='these tools represent a shift from bespoke RAG implementations to standardized,\nreusable, and often language-agnostic development paradigms.\n9.3 Deployment Considerations for Production RAG\nAlthough the RAG pattern is conceptually simple, production systems face multi-\ndimensional constraints, including retrieval quality, scalability, token/context budgets,\nlatency, security/governance, and evaluation/observability (Microsoft 2026b). In this'),
 Document(metadata={'_additional': {'explainScore': '\nHybrid (Result Set keyword,bm25) Document 81d4300c-f562-424b-91fa-b805846e45e3: original score 2.434456, normalized score: 0

### Building a RetrievalQA RAG Chain

RetrievalQA from LangChain connects a retriever and an LLM into a single chain that handles the full question-answering loop automatically.

When you call invoke on this chain with a question it performs three steps. First it sends the question to the retriever which calls Weaviate and gets back the most relevant chunks. Then it builds a prompt that contains all those chunks as context followed by the question. Finally it passes that prompt to the LLM and returns the generated text as the answer.

chain_type equals "stuff" means all retrieved chunks are concatenated together into one block and stuffed directly into the prompt. This is the simplest approach and works well when the combined chunk text fits within the LLM context window.

In [38]:
from langchain.chains import RetrievalQA

# Build the hybrid RAG chain
# llm: the TinyLlama model wrapped as a LangChain LLM
# chain_type="stuff": concatenate all retrieved chunks into one prompt
# retriever: the Weaviate hybrid retriever that fetches relevant chunks
hybrid_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever
)

In [40]:
# Run a question through the full RAG chain
# Weaviate retrieves the 2 most relevant chunks, TinyLlama generates an answer from them
result1 = hybrid_chain.invoke("What is Retrieval Augmented Generation?")
print(result1)

{'query': 'What is Retrieval Augmented Generation?', 'result': "Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.\n\n𝐿𝐿out\n(d) Parametric fusion\nQuery 𝑞𝑞: What is the tallest \nmountain in the world?\nRetriever\nTop-k LoRA modules\nGenerator\nAnswer 𝑦𝑦: Mount Everest\n𝑥𝑥 𝐿𝐿1 𝐿𝐿2𝐿𝐿in 𝐿𝐿n 𝑦𝑦\n⨁\nMerged \nLoRA\nmodule\n𝐿𝐿out\n(c) Latent fusion\nQuery 𝑞𝑞: What is the tallest \nmountain in the world?\nRetriever\nTop-k chunk texts\n𝑧𝑧1, 𝑧𝑧2, … , 𝑧𝑧𝑘𝑘\nGenerator\nAnswer 𝑦𝑦: Mount Everest\n𝑥𝑥 𝐿𝐿1 𝐿𝐿2𝐿𝐿in 𝐿𝐿n 𝑦𝑦\nEncoderℎ𝑧𝑧1, ℎ𝑧𝑧2, … , ℎ𝑧𝑧𝑘𝑘\n⨁ ⨁ ⨁\n(b) Logits-based fusion\nQuery 𝑞𝑞: What is the tallest \nmountain in the world?\nRetriever\n\nV1/2024.FINDINGS-EMNLP.622\nRu D, Qiu L, Hu X, et al (2024) Ragchecker: A fine-grained framework for diagnos-\ning retrieval-augmented generation. In: Advances in Neural Information Processing\nSystems 38 (NeurIPS)\nSaad-Falcon J, Khattab O, Po

In [ ]:
# Ask a different question through the same chain
# Passing the query as a dict with the "query" key is equivalent to passing a plain string
query = "What is Abstractive Question Answering?"
response = hybrid_chain.invoke({"query": query})

### LCEL Style RAG Chain

LangChain Expression Language (LCEL) is a newer way to compose chains using the pipe operator. Instead of calling RetrievalQA.from_chain_type you build the chain by piping steps together explicitly.

RunnableParallel runs multiple runnables in parallel and combines their outputs into a dictionary. Here it runs the retriever for the context key and passes the question through unchanged using RunnablePassthrough for the question key. The resulting dictionary is then piped to a prompt template and finally to the LLM.

This approach gives more explicit control over the prompt structure and is the recommended modern style in LangChain.

In [43]:
from langchain_core.prompts import ChatPromptTemplate

# Import ChatPromptTemplate.
# It creates prompts in a chat format, where messages are assigned roles
# such as "system", "human", and "assistant".
# Most chat-based LLMs (Llama, GPT, Qwen, etc.) expect prompts in this format.

system_prompt = (
    # This is the instruction given to the LLM before the user's question.
    # It tells the model how it should behave.

    "Use the given context to answer the question. "

    # If the retrieved context does not contain the answer,
    # instruct the model not to guess.
    "If you don't know the answer, say you don't know. "

    # Keep the response short.
    "Use three sentence maximum and keep the answer concise. "

    # Placeholder where the retrieved documents will be inserted.
    "Context: {context}"
)

In [44]:
# Create the complete chat prompt.
prompt = ChatPromptTemplate.from_messages(

    [
        # System message
        # Gives permanent instructions to the LLM.
        ("system", system_prompt),

        # Human message
        # Placeholder where the user's query will be inserted.
        ("human", "{query}"),
    ]
)

In [45]:
from langchain.prompts import PromptTemplate

# Import PromptTemplate.
# It creates a plain text prompt using placeholders.
# Unlike ChatPromptTemplate, it does not separate messages into roles.

template = """

# Instruction for the LLM.
Use the following pieces of context to answer the question at the end.

# Tell the model not to hallucinate.
If you don't know the answer, just say that you do not have the relevant information needed to provide a verified answer, don't try to make up an answer.

# Tell the model what writing style to follow.
When providing an answer, aim for clarity and precision.
Position yourself as a knowledgeable authority on the topic,
but also be mindful to explain the information in a manner
that is accessible and comprehensible to those without a technical background.

# Force the model to always end with this sentence.
Always say "Do you have any more questions pertaining to this instrument?"
at the end of the answer.

# Placeholder where retrieved documents will be inserted.
{context}

# Placeholder where the user's question will be inserted.
Question: {question}

# Tell the LLM where to begin generating.
Helpful Answer:
"""

# Create the PromptTemplate object.
prompt = PromptTemplate.from_template(template)

In [46]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

# Set up the RAG chain using LCEL pipe syntax
# Step 1: RunnableParallel runs retriever and passthrough simultaneously
#   context key: retriever fetches relevant chunks from Weaviate and returns their text
#   question key: RunnablePassthrough forwards the original question unchanged
# Step 2: the combined dict is piped into a prompt template that formats context and question
# Step 3: the formatted prompt is piped into the LLM which generates the answer
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()} |
    prompt |
    llm
)

In [48]:
# Run the LCEL chain with a question string
# The chain automatically retrieves context from Weaviate and generates an answer
query = "what is RAG?"
response = rag_chain.invoke("what is RAG?")

In [50]:
# Inspect the full response object
response

'\n\n# Instruction for the LLM.\nUse the following pieces of context to answer the question at the end.\n\n# Tell the model not to hallucinate.\nIf you don\'t know the answer, just say that you do not have the relevant information needed to provide a verified answer, don\'t try to make up an answer.\n\n# Tell the model what writing style to follow.\nWhen providing an answer, aim for clarity and precision.\nPosition yourself as a knowledgeable authority on the topic,\nbut also be mindful to explain the information in a manner\nthat is accessible and comprehensible to those without a technical background.\n\n# Force the model to always end with this sentence.\nAlways say "Do you have any more questions pertaining to this instrument?"\nat the end of the answer.\n\n# Placeholder where retrieved documents will be inserted.\n[Document(metadata={}, page_content=\'demonstrations which are concatenated into inputs (Wang et al. 2022; Li et al. 2023d;\\nHuang et al. 2023a), generators in RAG can 

## Category 5  Reranking With Cohere

### What Reranking Is and Why It Improves RAG

The retrieval step in a RAG pipeline is a first-pass approximation. BM25 and dense vector search are fast but they use relatively simple scoring methods: BM25 counts keyword overlaps and vector search measures cosine similarity between single-pass embeddings. Neither method reads the query and document together; they score each independently.

A reranker uses a cross-encoder architecture. It reads the query and a candidate document as a pair in a single forward pass through the model. This joint reading lets the model understand exactly how the query and document relate to each other, producing much more accurate relevance scores than first-pass retrieval.

The typical pipeline is called retrieve-then-rerank. The retriever fetches a broad set of candidates quickly (top 10 or 20). The reranker then re-scores all candidates and returns only the top k. The LLM receives only the highest-quality documents as context.

### Why Cohere

Cohere provides a dedicated reranking API called rerank-english-v2.0 that is specifically trained to judge the relevance of a document to a query. It is fast, accurate, and accessible via a simple API call without needing to download any model weights locally. You get a free tier that is sufficient for development and experimentation.

In [ ]:
# Install the Cohere Python client library
# This is needed because CohereRerank calls the Cohere Rerank API internally
!pip install cohere

In [53]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import CohereRerank

### Setting Up the Cohere Reranker

CohereRerank is a LangChain document compressor. The term compressor here does not mean making text shorter. It means compressing the list of retrieved candidates down to only the most relevant ones by rescoring and filtering them.

The base_retriever (Weaviate hybrid) fetches an initial set of candidates. CohereRerank then sends those candidates together with the original query to the Cohere Rerank API. Cohere scores every candidate and returns them in a new order. ContextualCompressionRetriever combines these two steps into a single retriever interface so the rest of the chain does not need to change.

In [55]:
load_dotenv()

compressor = CohereRerank(
    cohere_api_key=os.getenv("cohere_api_key"),
    model="rerank-v3.5",
    top_n=3
)

In [56]:
# Wrap the Weaviate hybrid retriever with the Cohere reranker
# base_compressor: the reranker that rescores candidates returned by the base retriever
# base_retriever: the Weaviate hybrid retriever that fetches the initial candidates
# When invoke is called on compression_retriever it first calls Weaviate to get candidates
# and then passes those candidates to Cohere for reranking before returning the final list
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=retriever
)

### Testing the Reranker Directly

Before using the reranker inside a full RAG chain it is good practice to test it standalone. get_relevant_documents calls the full retrieve-then-rerank pipeline and returns the final reranked list of documents. Printing the results lets you verify that the reranker changed the ordering compared to the raw Weaviate results.

In [57]:
# Define the query to test the compression retriever
user_query = "What is Abstractive Question Answering?"

# get_relevant_documents triggers the full pipeline:
#   1. Weaviate hybrid search retrieves the initial candidates
#   2. Cohere Rerank rescores all candidates against the query
#   3. The reranked list is returned with the most relevant document first
compressed_docs = compression_retriever.get_relevant_documents(user_query)

# Print the relevant documents from using the embeddings and reranker
print(compressed_docs)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_11924\2086512010.py:8: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  compressed_docs = compression_retriever.get_relevant_documents(user_query)


[Document(metadata={'relevance_score': 0.38034514}, page_content='from noisy retrievals.\n6.4 Question Answering\nTask Characteristics.Question Answering (QA) is a fundamental task in NLP\nthat involves building systems capable of automatically answering human questions\nin natural language. QA systems can be broadly classified into two categories: open-\ndomain, where the system answers questions about virtually anything, and closed-\ndomain, focusing on a specific area of knowledge. Due to the page limits, this paper'), Document(metadata={'relevance_score': 0.048482586}, page_content='edge amplification in multi-document question answering. In: Proceedings of the\n2024 Conference on Empirical Methods in Natural Language Processing (EMNLP),\n53')]


### Building the Full RAG Chain With Reranking

Now we replace the plain hybrid retriever with the compression_retriever (hybrid retrieval plus Cohere reranking) inside a RetrievalQA chain. The LLM now receives documents that have been scored and ordered by a dedicated cross-encoder model rather than the approximate first-pass ranking from Weaviate alone.

This two-stage pipeline: hybrid retrieval for speed followed by cross-encoder reranking for precision, is the standard pattern used in production RAG systems.

In [58]:
# Build the reranked RAG chain
# Everything is identical to the earlier hybrid_chain except the retriever
# Using compression_retriever means every query goes through:
#   1. Weaviate hybrid search (BM25 plus dense vector, alpha=0.5)
#   2. Cohere Rerank cross-encoder rescoring
#   3. LLM answer generation from the top reranked chunks
hybrid_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=compression_retriever
)

In [59]:
# Run a question through the full reranked RAG chain
response = hybrid_chain.invoke("What is Abstractive Question Answering?")

In [60]:
# Print the generated answer from the reranked RAG pipeline
print(response.get("result"))

Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

from noisy retrievals.
6.4 Question Answering
Task Characteristics.Question Answering (QA) is a fundamental task in NLP
that involves building systems capable of automatically answering human questions
in natural language. QA systems can be broadly classified into two categories: open-
domain, where the system answers questions about virtually anything, and closed-
domain, focusing on a specific area of knowledge. Due to the page limits, this paper

edge amplification in multi-document question answering. In: Proceedings of the
2024 Conference on Empirical Methods in Natural Language Processing (EMNLP),
53

Question: What is Abstractive Question Answering?
Helpful Answer: Abstractive Question Answering (AQA) is a type of question answering (QA)
